<a href="https://colab.research.google.com/github/hebamuh68/eyouth/blob/main/week_5/projects/realtime_ai_inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Real-Time Inference: Camera to Prediction (Google Colab)

This notebook runs Python on Google's servers while your **webcam is on your computer**. The first code cell builds a bridge so frames from the browser are sent to Python.

**Before Step 2:** you must place **`keras_model.h5`** and **`labels.txt`** in `/content/`. Use the **Upload model files** cell below (easiest), or the Files sidebar (folder icon) → Upload.

**Optional:** If imports fail, run the install cell below, then use Runtime → Restart runtime.

In [1]:
# Optional: run once if tf_keras or PIL are missing
!pip install -q tf_keras pillow

## Upload model files (run before Step 2)

Export **`keras_model.h5`** and **`labels.txt`** from Teachable Machine (Image project → Export Model → TensorFlow → Keras). Run the next cell and choose both files from your computer.

Or use the left sidebar folder icon → **Upload** and drop both files into **`/content/`** (top level, not inside another folder). Names must match exactly: `keras_model.h5` and `labels.txt`.

In [2]:
from google.colab import files
import os

print("Select keras_model.h5 and labels.txt (Ctrl or Cmd + click to pick both).")
uploaded = files.upload()
for name in uploaded:
    print("Saved:", name, "(" + str(len(uploaded[name])) + " bytes)")
print("Contents of /content/:", sorted(os.listdir("/content")))

Select keras_model.h5 and labels.txt (Ctrl or Cmd + click to pick both).


Saving keras_model.h5 to keras_model.h5
Saving labels.txt to labels.txt
Saved: keras_model.h5 (43360 bytes)
Saved: labels.txt (24 bytes)
Contents of /content/: ['.config', '.ipynb_checkpoints', 'content', 'keras_model.h5', 'labels.txt', 'realtime_ai_inference_colab.ipynb', 'sample_data']


## Step 1 — Camera bridge (run once)

Run this cell first. It defines `video_stream()` and `video_frame()` so later code can read webcam frames from the browser.

**Camera:** Colab uses **your local browser’s** webcam (not the server). Use **Chrome or Edge**, click the lock/camera icon if prompted and **Allow**, close other apps using the camera, and run on a machine that actually has a webcam. School-managed laptops sometimes block camera access.

In [3]:
import base64
import numpy as np
from io import BytesIO
from PIL import Image as PILImage
from tf_keras.models import load_model
from google.colab.output import eval_js
from IPython.display import display, Javascript


def video_stream():
    """Shows the webcam preview in the browser and sends frames to Python."""
    js = Javascript('''
    var video; var div = null; var stream; var captureCanvas; var labelElement;
    var pendingResolve = null; var shutdown = false;
    var cameraFailed = false;

    function removeDom() {
      if (stream) {
        stream.getTracks().forEach(track => track.stop());
        stream = null;
      }
      if (div) {
        div.remove();
        div = null;
      }
      video = undefined;
      captureCanvas = undefined;
      labelElement = undefined;
    }

    function onAnimationFrame() {
      if (!shutdown) window.requestAnimationFrame(onAnimationFrame);
      if (pendingResolve) {
        var result = "";
        if (!shutdown && !cameraFailed && video && captureCanvas) {
          try {
            captureCanvas.getContext('2d').drawImage(video, 0, 0, 224, 224);
            result = captureCanvas.toDataURL('image/jpeg', 0.8);
          } catch (e) { result = ""; }
        }
        var lp = pendingResolve; pendingResolve = null; lp(result);
      }
    }

    async function createDom() {
      if (div !== null) return;
      try {
        stream = await navigator.mediaDevices.getUserMedia({
          video: { width: { ideal: 224 }, height: { ideal: 224 } }
        });
      } catch (e) {
        cameraFailed = true;
        div = document.createElement('div');
        div.style.border = '2px solid #a00';
        div.style.padding = '10px';
        div.style.maxWidth = '400px';
        div.style.background = '#fff5f5';
        div.style.fontSize = '14px';
        div.innerText = 'Camera error: ' + e.name + '. Allow camera for this site (lock icon in the address bar), use Chrome/Edge, and a device with a working webcam. If the error is NotFoundError, no camera is available or it is in use by another app.';
        document.body.appendChild(div);
        return;
      }
      div = document.createElement('div');
      div.style.border = '2px solid black'; div.style.padding = '10px';
      div.style.width = '250px'; div.style.background = 'white';
      video = document.createElement('video');
      video.style.display = 'block'; video.width = 224; video.height = 224;
      video.autoplay = true; video.playsInline = true;
      div.appendChild(video);
      video.srcObject = stream;
      labelElement = document.createElement('div');
      labelElement.style.fontSize = '18px'; labelElement.style.fontWeight = 'bold';
      labelElement.style.marginTop = '10px'; labelElement.innerText = "Initializing...";
      div.appendChild(labelElement);
      document.body.appendChild(div);
      captureCanvas = document.createElement('canvas');
      captureCanvas.width = 224; captureCanvas.height = 224;
      window.requestAnimationFrame(onAnimationFrame);
    }

    async function streamFrame(label) {
      if (shutdown) return "";
      if (div === null) await createDom();
      if (cameraFailed) return "";
      if (labelElement) labelElement.innerText = label;
      return new Promise(resolve => { pendingResolve = resolve; });
    }
    ''')
    display(js)


def video_frame(label):
    return eval_js('streamFrame("{}")'.format(label))

## Step 2 — Load model and run live predictions

Interrupt the cell (stop button or Runtime → Interrupt execution) when you are done. The `finally` block releases the camera UI.

In [4]:
# 1. Load the model and labels
import os

MODEL_PATH = "/content/keras_model.h5"
LABELS_PATH = "/content/labels.txt"

if not os.path.isfile(MODEL_PATH) or not os.path.isfile(LABELS_PATH):
    print("keras_model.h5 present:", os.path.isfile(MODEL_PATH))
    print("labels.txt present:", os.path.isfile(LABELS_PATH))
    print("Files in /content/:", [f for f in os.listdir("/content") if not f.startswith(".")])
    raise FileNotFoundError(
        "Upload keras_model.h5 and labels.txt to /content/ first (run the Upload cell, or use the Files sidebar)."
    )

try:
    model = load_model(MODEL_PATH, compile=False)
    class_names = open(LABELS_PATH, "r").readlines()
    print("Model loaded.")
except Exception as e:
    print(f"Error loading files: {e}")
    raise

# 2. Start the camera UI
video_stream()
label = "Waiting for camera..."

try:
    while True:
        js_reply = video_frame(label)
        if not js_reply:
            break

        header, encoded = js_reply.split(",", 1)
        binary = base64.b64decode(encoded)
        image = PILImage.open(BytesIO(binary)).convert("RGB")
        image_np = np.asarray(image, dtype=np.float32)
        normalized_image = (image_np.reshape(1, 224, 224, 3) / 127.5) - 1

        prediction = model.predict(normalized_image, verbose=0)
        index = int(np.argmax(prediction))
        conf = prediction[0][index] * 100

        label = f"{class_names[index].strip()} ({conf:.1f}%)"

except KeyboardInterrupt:
    print("Stopped by user.")
finally:
    eval_js('shutdown = true; removeDom()')

Model loaded.


<IPython.core.display.Javascript object>

## Optional — Quick model check (no camera)

Run this in a **fresh runtime** if you only want to verify the files load. Skip if you already ran the cells above in the same session.

In [5]:
import os
from tf_keras.models import load_model

MODEL_PATH = "/content/keras_model.h5"
LABELS_PATH = "/content/labels.txt"
if not os.path.isfile(MODEL_PATH) or not os.path.isfile(LABELS_PATH):
    raise FileNotFoundError("Upload keras_model.h5 and labels.txt to /content/ first.")

model = load_model(MODEL_PATH, compile=False)
class_names = open(LABELS_PATH, "r").readlines()
print("Input shape:", model.input_shape)
print("Labels:", [c.strip() for c in class_names])

Input shape: (None, 224, 224, 3)
Labels: ['Thumbs Up', 'Thumbs Down']
